基础的plt的应用

In [ ]:
def function_1(x):
     return 0.01*x**2 + 0.1*x
import numpy as np
import matplotlib.pylab as plt
x=np.arange(0.0,20.0,0.1)
y=function_1(x)
plt.xlabel("x")
plt.ylabel("y")
plt.plot(x,y)
plt.show()

下面再用nmupy进行的
来看代码的梯度，这里应该是数值微分的方法


In [ ]:
import sys, os
sys.path.append(os.pardir)
import numpy as np
from common.functions import softmax, cross_entropy_error
from common.gradient import numerical_gradient

class simpleNet:
    def __init__(self):
        self.W=np.random.randn(2,3)
    def predict(self,x):
        return np.dot(x, self.W)
    def loss(self,x,t):
        z=self.predict(x)
        y=softmax(z)
        loss=cross_entropy_error(y,t)
        return loss


net=simpleNet()
print(net.W)
x=np.array([0.6,0.9])
p=net.predict(x)
print(p)
np.argmax(p)
t = np.array([0, 0, 1]) # 正确解标签
net.loss(x, t)

In [ ]:
def f(W):
    return net.loss(x,t)
dW=numerical_gradient(f, net.W)
print(dW)

实现mini-batch,输入的ssl可以避免身份认证。在这里还人为的创造了一个twolayernet

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
from dataset.mnist import load_mnist
from ch04.two_layer_net import TwoLayerNet
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)
train_loss_list = []
# 超参数
iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)
for i in range(iters_num):
# 获取 mini-batch
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]
    # 计算梯度
    grad = network.numerical_gradient(x_batch, t_batch)
    # grad = network.gradient(x_batch, t_batch) # 高速版 !
    # 更新参数
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]
        # 记录学习过程
        loss = network.loss(x_batch, t_batch)
        train_loss_list.append(loss)


In [ ]:
class SGD:
    
    def __init__(self, lr=0.01):
        self.lr = lr
    def update(self, params, grads):
        for key in params.keys():
            params[key] -= self.lr * grads[key]
# pytorch去调用类对象的时候
# network=...
# optimizer=...



SGD 低效的根本原因是，梯度的方向并没有指向最小值的方
所以进行其他优化
下面是Momentum
在物体不受任何力时，该项承担使物体逐渐减
速的任务（α 设定为 0.9 之类的值），对应物理上的地面摩擦或空气阻力。下
面是 Momentum 的代码实现

In [ ]:
def update(self, params, grads):
    if self.v is None:
        self.v = {}
        for key, val in params.items():
           self.v[key] = np.zeros_like(val)
    for key in params.keys():
        self.v[key] = self.momentum * self.v[key] - self.lr * grads[key]
        params[key] += self.v[key]

AdaGrad
缺点是一直更新，会累计所以梯度的平方和，最后会近于0
可以用RMSprop方法

权重初始值的设定
不能设置成相同值，不同神经元应该有自己的特征，自己的关注点，
如果初始值一样，将造成神经元只会学习同一个特征，无法打破对称性
Relu有自己的He,sigmoid有自己的Xaavier ，区别在于标准差
目的在于为了尽力维持每一层之间的方差稳定，反应了特征值的广度
还有也是为了防止梯度消失，比如sigmoid


超参数最优化的一种方法
实践经验而已
这里介绍的超参数的最优化方法是实践性的方法。不过，这个方
法与其说是科学方法，倒不如说有些实践者的经验的感觉。在超
参数的最优化中，如果需要更精炼的方法，可以使用贝叶斯最优
化（Bayesian optimization）。贝叶斯最优化运用以贝叶斯定理为中
心的数学理论，能够更加严密、高效地进行最优化。详细内容请
参 考 论 文“Practical Bayesian Optimization of Machine Learning 
Algorithms”[16] 等

In [ ]:
if i % iter_per_epoch == 0:
    train_acc = network.accuracy(x_train, t_train)
    test_acc = network.accuracy(x_test, t_test)
    train_acc_list.append(train_acc)
    test_acc_list.append(test_acc)
    epoch_cnt += 1
if epoch_cnt >= max_epochs:
       break

CNN
与上述神经网络不同的是
它多了卷积层和池化层
全连接层的困境，忽视了一个图像的三维，因为它的原始输入是一列
但是卷积层可以保持不变
下面截取一段文字
有 时 将 卷 积 层 的 输 入 输 出 数 据 称 为 特 征 图（feature
map）。其中，卷积层的输入数据称为输入特征图（input feature map），输出
数据称为输出特征图（output feature map

In [ ]:
from dataset.mnist import load_mnist,shuffle_dataset
(x_train,t_train),(x_test,t_test)=load_mnist(normalize=True, one_hot_label=True)
# 打乱数据
x_train,t_train=shuffle_dataset(x_train,t_train)
validation_rate=0.2
validation_size=int(x_train.shape[0]*validation_rate)
# 然后就是对训练集进行划分，划分出验证集和训练集

CNN处理的是四维数据
数量，高，长，通道

下面是训练数据和验证数据的分割
训练数据是为了参数的学习
验证数据是为了超参数的调整
测试数据是最后一考


池化层
鲁棒性：对微小偏差的抗拒能力